In [83]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("BoulderResults.csv")

# Sanity check on load
print(df.shape)
print(df.head())

# Per-boulder best score and % of best
# Group by the boulder identity (event + round + boulder number),
# Find the max score anyone achieved on that specific boulder
df["Best_Score"] = df.groupby(["Event_ID", "Round", "Boulder_Num"])["Score"].transform("max")

# % of best to avoid skewing with difficult rounds / easy rounds
# Guard against division by zero for the rare boulder where everyone scored 0
df["Pct_Of_Best"] = df.apply(
    lambda row: row["Score"] / row["Best_Score"] if row["Best_Score"] > 0 else 0.0,
    axis=1
)

(2596, 6)
  Event_ID      Round  Boulder_Num Athlete_ID  Score  Topped
0    HAC23  Semifinal            1     PJENFT   10.0       0
1    HAC23  Semifinal            2     PJENFT    9.6       0
2    HAC23  Semifinal            3     PJENFT   24.6       1
3    HAC23  Semifinal            4     PJENFT   24.5       1
4    HAC23  Semifinal            1      JCHON    0.0       0
  Event_ID      Round  Boulder_Num Athlete_ID  Score  Best_Score  Pct_Of_Best
0    HAC23  Semifinal            1     PJENFT   10.0        10.0     1.000000
1    HAC23  Semifinal            2     PJENFT    9.6        24.3     0.395062
2    HAC23  Semifinal            3     PJENFT   24.6        24.8     0.991935
3    HAC23  Semifinal            4     PJENFT   24.5        25.0     0.980000
4    HAC23  Semifinal            1      JCHON    0.0        10.0     0.000000
5    HAC23  Semifinal            2      JCHON    0.0        24.3     0.000000
6    HAC23  Semifinal            3      JCHON   24.6        24.8     0.991935


First, to narrow the search I've decided to consider athletes in the semi-final round straight from their semi-final progression rate. Reason being, I'm unsure of the style of qualification boulders since they aren't documents and there are 2 groups. Group A and Group B of which the semi-finalists are selected from. This is difficult to model and so I've decided to purely use progression rates to determine our predicted semi-finalists. Whilst the full list may differ from competition to competition, the main group of semi-finalists (higher scorers) don't differ to drastically.

# Semi-Finalists Note
I utilise a formula to obtain the athletes with the highest 'progression score'

The formula I will use per athlete is : (SemiFinals + k * prior rate)/(Attended + k)

The reason for this is to account for athletes with fewer appearances and to fairly compare athletes to obtain our semi finals list. This is because less appearances may unfairly assign a progression rate of 0% or 100% which would cause the semi finals prediction to be inaccurate.

As such I've applied Bayesian shrinkage to avoid this by blending each athlete's rate of semi finals progression with the population's rate of semi finals progression. This is a form of empirical Bayes estimation utilised to help in ranking these athletes for the semi finals round.

The base rate is calculated as the rate of semi final progression without inspecting the individual climbers (total semifinals / total appearances)

k represents a number of hypothetical extra appearances, assumed to occur at the population base rate, added to every athlete's record. It functions as a threshold for how much real data is needed before an athlete's own rate is trusted over the population average, rather than the average being trusted over their data. Which aids in the issue of giving inaccurate scores to more absent climbers.

In this case I choose k = 5. This is because this value is close to the median (4) of the data set and so should help adjust those athletes with lower attendances. I chose 5 instead of 4 so the effect can be more obvious.



Note on Bayesian Shrinkage.

This applies here because, with an athlete that has a low attendance rate, we could use the population's conversion rate rather than the data estimated conversion rate in order to properly gauge how much of a chance they should have since we do not have signifcant evidence to give them a 0.

However, athletes with many attendances provide enough data for us to use their data to estimate their conversion rate.

In [84]:
# Start finding our semi-finalists
# I utilise a formula to obtain the athletes with the highest 'progression score'
# The formula I will use per athlete is : (SemiFinals + k * prior rate)/(Attended + k)
# The reason for this formula is b
# In this case I choose k = 5.

Qualification = pd.read_csv("Qual_Boulder_Results.csv")

# Obtain the prior rate
base_rate = Qualification["SemiFinals"].sum()/ Qualification["Attended"].sum()

def blended_rate(row, k):
  return (row["SemiFinals"] + k * base_rate) / (row["Attended"] + k)

# Now we have a table for k = 3,4,5

k3 = Qualification[["Athlete_ID", "SemiFinals", "Attended"]].copy()
k4 = Qualification[["Athlete_ID", "SemiFinals", "Attended"]].copy()
k5 = Qualification[["Athlete_ID", "SemiFinals", "Attended"]].copy()

k3["Blended_Rate"] = k3.apply(lambda row: blended_rate(row, 3), axis=1)
k4["Blended_Rate"] = k4.apply(lambda row: blended_rate(row, 4), axis=1)
k5["Blended_Rate"] = k5.apply(lambda row: blended_rate(row, 5), axis=1)

k3 = k3.sort_values("Blended_Rate" ,ascending=False).reset_index(drop=True)
k4 = k4.sort_values("Blended_Rate" ,ascending=False).reset_index(drop=True)
k5 = k5.sort_values("Blended_Rate" ,ascending=False).reset_index(drop=True)

print(k3.head(24))
print(k4.head(24))
print(k5.head(24))



      Athlete_ID  SemiFinals  Attended  Blended_Rate
0        SANRAKU          12        12      0.863500
1      TNARASAKI          11        11      0.853750
2           DLEE          11        11      0.853750
3       MSCHALCK          11        12      0.796833
4        APEHARC          11        12      0.796833
5      RKAWAMATA           5         5      0.744062
6       SAMAGASA          10        12      0.730167
7         MMILNE           7         8      0.722955
8         CDUFFY           9        11      0.710893
9          JCHON           6         7      0.695250
10   JMACDOUGALL           8        10      0.688654
11       DAKHTAR           8        10      0.688654
12          YPAN           8        10      0.688654
13      TROBERTS           5         6      0.661389
14        PJENFT           6         8      0.632045
15      SRICHARD           6         8      0.632045
16         KDOHI           4         5      0.619062
17     MNARASAKI           7        10      0.

We see the top 24 do not change as we differ from k=3,4,5. As a result we choose 5.

Further I choose to have a table consisting of the top 28 athletes because some athletes may have off seasons or good seasons / months. So I will allow for this leniency.


In [85]:
# We also include the top 28 to allow for close calls and check for accuracy

SemifinalsPool = k5.nlargest(28, "Blended_Rate")

Semifinals = k5.nlargest(24, "Blended_Rate")

print(SemifinalsPool)
print(Semifinals)




      Athlete_ID  SemiFinals  Attended  Blended_Rate
0        SANRAKU          12        12      0.799265
1      TNARASAKI          11        11      0.786719
2           DLEE          11        11      0.786719
3       MSCHALCK          11        12      0.740441
4        APEHARC          11        12      0.740441
5       SAMAGASA          10        12      0.681618
6         CDUFFY           9        11      0.661719
7         MMILNE           7         8      0.660577
8      RKAWAMATA           5         5      0.658750
9    JMACDOUGALL           8        10      0.639167
10       DAKHTAR           8        10      0.639167
11          YPAN           8        10      0.639167
12         JCHON           6         7      0.632292
13      TROBERTS           5         6      0.598864
14        PJENFT           6         8      0.583654
15      SRICHARD           6         8      0.583654
16     MNARASAKI           7        10      0.572500
17         KDOHI           4         5      0.

I want to validate the accuracy of semi finals prediction

In [86]:
# Obtaining accuracy % of semifinalists

BoulderResults = pd.read_csv("BoulderResults.csv")

# First, some functions to help with the calculations

def semifinalists_for_event(event_id, BoulderResults):
    return set(BoulderResults[(BoulderResults["Event_ID"] == event_id) &
                           (BoulderResults["Round"] == "Semifinal")]["Athlete_ID"])
    # Gives the set of results for a particular event in the semi finals and the athlete IDs

def compare_pool_to_event(Semifinals, event_id, results_df): # Make the actual comparison calculations
  # actual semi finals
  actual = semifinalists_for_event(event_id, results_df)

  predicted = set(Semifinals)

  overlap = predicted & actual # Gives the athletes that the model correctly predicts to enter

  return {
        "Event_ID": event_id,
        "Pool_Size": len(predicted),        # Number of athletes in the prediction pool
        "Actual_SF_Size": len(actual),       # Number of semi finalists
        "Overlap": len(overlap),             # Correct predictions

        # Precision calculation, which of my prediction pool enterred the semi finals
        "Precision": round(len(overlap) / len(predicted), 3) if predicted else None,

        # Recall calculation, out of the actual semi finalists, how many did I predicted
        "Recall": round(len(overlap) / len(actual), 3) if actual else None,

        # Athletes who made semifinals but my model didn't catch
        "Missed": sorted(actual - predicted),

        # Athletes that were predicted to make finals that didn't actual make finals
        "Extra": sorted(predicted - actual),
  }

# ----------------------------------------------------------------------------------------

# Actual calculations

predicted_SF = k5.head(24)["Athlete_ID"].tolist()

events_2026 = ["KEQ26", "BER26", "MAD26", "PRA26", "IBK26"]

# Run comparisons for each event in 2026

rows = [compare_pool_to_event(predicted_SF, eid, BoulderResults) for eid in events_2026]

# Data frame for comparisons

SFComparisondf = pd.DataFrame(rows)

print(SFComparisondf[["Event_ID", "Pool_Size", "Actual_SF_Size", "Overlap", "Precision", "Recall"]])


# ----------------------------------------------------------------------------------------

# Retrieve overall comparison across the whole of 2026


total_overlap = SFComparisondf["Overlap"].sum()
total_pool = SFComparisondf["Pool_Size"].sum()
total_actual = SFComparisondf["Actual_SF_Size"].sum()


print("\nOverall precision across all 2026 events:", round(total_overlap / total_pool, 3))
print("Overall recall across all 2026 events:", round(total_overlap / total_actual, 3))

  Event_ID  Pool_Size  Actual_SF_Size  Overlap  Precision  Recall
0    KEQ26         24              24       17      0.708   0.708
1    BER26         24              24       16      0.667   0.667
2    MAD26         24              24       16      0.667   0.667
3    PRA26         24              24       17      0.708   0.708
4    IBK26         24              24       16      0.667   0.667

Overall precision across all 2026 events: 0.683
Overall recall across all 2026 events: 0.683
